# Pneumonia Detection from Chest X-Ray Images
## Final Capstone Notebook — Three-Class Classification

**Classes:** `Pneumonia` (Lung Opacity) · `Normal` · `Not Normal / No Lung Opacity`

This notebook extends the interim binary-classification baseline to the full
three-class problem required for the final submission. It follows the
structure: Data Overview → EDA → Preprocessing → CNN from scratch →
Transfer Learning (VGG16, ResNet50) → Enhanced architectures → Fine-tuning →
Model comparison → Best model selection → Serialization → Inference →
Deployment prep.


## Business Context

Pneumonia is a leading cause of morbidity and mortality worldwide, particularly
among children under five and elderly populations. Diagnosis relies heavily on
radiologist interpretation of chest X-rays, but radiologist availability is
limited in many settings, and factors such as fatigue, case volume, and
inter-reader variability can affect diagnostic consistency.

A deep-learning-based classifier is explored here as a **clinical
decision-support tool** — intended to help flag potentially abnormal cases
for prioritized review, not to replace radiologist diagnosis. Unlike the
interim binary version of this project, the final model distinguishes three
outcomes:

1. **Pneumonia** — lung opacity consistent with pneumonia
2. **Normal** — no abnormal finding
3. **Not Normal / No Lung Opacity** — an abnormality is present, but it is
   *not* pneumonia. This class is clinically important because such findings
   can visually resemble pneumonia-consistent opacities, and collapsing it
   into either of the other two classes (as the interim notebook did for
   scoping reasons) would hide a realistic and clinically relevant source of
   confusion.


## Objective

Build and compare a from-scratch CNN and multiple transfer-learning
architectures (VGG16, ResNet50 — carried forward from the interim notebook's
proposed next steps) for three-class chest X-ray classification, select a
best model using medically-relevant criteria (not accuracy alone), and
package it for deployment via Streamlit + Docker.


## Installing and Importing Libraries

In [ ]:
# Run once per environment (Colab/Kaggle). Comment out if already installed.
# !pip install pydicom -q


In [ ]:
import os
import random

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import cv2
import pydicom

import tensorflow as tf
import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Dense, Dropout, Flatten, Conv2D, MaxPooling2D,
    BatchNormalization, GlobalAveragePooling2D, Input
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.applications import VGG16, ResNet50
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg16_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet50_preprocess

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)

import warnings
warnings.filterwarnings("ignore")

print("TensorFlow version:", tf.__version__)
print("Num GPUs Available:", len(tf.config.list_physical_devices("GPU")))


In [ ]:
# Reproducibility: sets NumPy, backend, and Python random seeds together
RANDOM_STATE = 42
tf.keras.utils.set_random_seed(RANDOM_STATE)


---
# 1. Data Overview

**Why this step is needed:** before any modeling, we confirm the dataset's
shape, class definitions, and image storage format so downstream steps
(labeling, splitting, preprocessing) are grounded in verified facts rather
than assumptions.

**What is being done:** this dataset is the RSNA Pneumonia Detection
Challenge set. Two metadata files are loaded — the detailed class-info
table (three-way diagnostic label) and the bounding-box/target table.
Unlike the interim notebook, the final project uses the **full patient
population**, not a 500-image subsample, subject to compute limits on your
Colab/Kaggle runtime (adjust `SAMPLE_SIZE` below if needed — set to `None`
to use everything).

In [ ]:
# =====================================================================
# CONFIGURATION - update paths for your Drive / Kaggle structure
# =====================================================================

# Uncomment if using Google Colab
# from google.colab import drive
# drive.mount('/content/drive')

DATA_DIR = "/content/drive/MyDrive/Pneumonia_Detection_Project/data"
IMAGE_DIR = f"{DATA_DIR}/stage_2_train_images"
CLASS_INFO_CSV = f"{DATA_DIR}/stage_2_detailed_class_info.csv"
TRAIN_LABELS_CSV = f"{DATA_DIR}/stage_2_train_labels.csv"

# Image sizing: 128x128 for the from-scratch CNN (matches interim baseline,
# keeps that model lightweight for a fair "baseline vs improvement" story);
# 224x224 for VGG16/ResNet50, since both were trained on ImageNet at that
# resolution and transfer quality degrades if inputs deviate far from it.
IMG_SIZE_CNN = 128
IMG_SIZE_TRANSFER = 224

RANDOM_STATE = 42

# Set to an integer (e.g. 3000) to subsample for faster iteration, or None
# to use the full ~26.7k patients. Recommend starting with a subsample to
# validate the pipeline end-to-end before committing to a full run.
SAMPLE_SIZE = None

TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.15, 0.15
assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-9


### Why this cell exists

Reading tens of thousands of individual files directly off a Google
Drive-mounted path (`/content/drive/...`) is very slow — each file open
carries significant FUSE/network latency, and a shuffled `tf.data`
pipeline reads files in essentially random order, which is the worst-case
access pattern for Drive. **An epoch that doesn't complete after an hour is
a symptom of this, not of the model or GPU** — the T4 sits idle waiting on
data.

**Fix:** copy the image folder to the Colab instance's local disk once,
then point `IMAGE_DIR` at the local copy for all training. If you have the
original Kaggle `.zip` of `stage_2_train_images` available (in Drive or
re-downloadable via the Kaggle API), copying/extracting that single zip is
far faster than copying 26k+ individual files — copy it as one big
sequential file rather than many small ones. The cell below handles both
cases: it uses a zip if you point it at one, otherwise falls back to a
direct (slower, but still far better than training-time random access)
folder copy.

In [ ]:
import shutil, zipfile, time

# ---- Option A (preferred if available): path to a zip of stage_2_train_images ----
LOCAL_ZIP_SOURCE = None  # e.g. "/content/drive/MyDrive/.../stage_2_train_images.zip"

# ---- Local destination (Colab instance disk - resets each session) ----
LOCAL_IMAGE_DIR = "/content/stage_2_train_images"

def materialize_images_locally():
    if os.path.isdir(LOCAL_IMAGE_DIR) and len(os.listdir(LOCAL_IMAGE_DIR)) > 0:
        print(f"Local copy already exists at {LOCAL_IMAGE_DIR} "
              f"({len(os.listdir(LOCAL_IMAGE_DIR))} files) — skipping copy.")
        return LOCAL_IMAGE_DIR

    os.makedirs(LOCAL_IMAGE_DIR, exist_ok=True)
    start = time.time()

    if LOCAL_ZIP_SOURCE is not None and os.path.exists(LOCAL_ZIP_SOURCE):
        print(f"Extracting {LOCAL_ZIP_SOURCE} -> {LOCAL_IMAGE_DIR} ...")
        with zipfile.ZipFile(LOCAL_ZIP_SOURCE, "r") as zf:
            zf.extractall(os.path.dirname(LOCAL_IMAGE_DIR))
    else:
        print(f"No zip provided — copying folder directly from {IMAGE_DIR}.")
        print("This reads every file individually from Drive; it is a one-time")
        print("cost but can still take a while for ~26.7k files.")
        shutil.copytree(IMAGE_DIR, LOCAL_IMAGE_DIR, dirs_exist_ok=True)

    elapsed = time.time() - start
    n_files = len(os.listdir(LOCAL_IMAGE_DIR))
    print(f"Done in {elapsed/60:.1f} min. {n_files} files now at {LOCAL_IMAGE_DIR}.")
    return LOCAL_IMAGE_DIR


# Run once per Colab session. Comment out after confirming it worked if you
# want to skip the check on re-runs within the same session.
IMAGE_DIR = materialize_images_locally()
print("IMAGE_DIR now points to:", IMAGE_DIR)


**While the full copy is running (or if you'd rather not wait for it right
now):** you can keep developing/validating the rest of this notebook
against a small `SAMPLE_SIZE` — reading a few thousand files from Drive
individually is slow-but-tolerable (minutes, not hours), whereas the full
26.7k-image training run is what actually needs the local copy. Suggested
workflow:

1. Set `SAMPLE_SIZE = 2000` (in the Configuration cell above) for now.
2. Re-run from the Configuration cell down through this notebook to
   validate every section end-to-end quickly.
3. Once validated, kick off the local copy above (or let it run in the
   background), set `SAMPLE_SIZE = None`, and do the real full-scale
   training run.

This avoids burning hours re-discovering a pipeline bug on the full
dataset — better to find bugs on a fast, small run first.

In [ ]:
# Load class info (one row per patient after de-duplication).
# The raw CSV has one row per bounding box, so Lung Opacity patients with
# multiple annotated opacities appear multiple times; since this is a
# classification (not detection) task, we need exactly one label per patient.
class_info_raw = pd.read_csv(CLASS_INFO_CSV)
print("Raw class-info shape (one row per bounding box):", class_info_raw.shape)

class_info = class_info_raw.drop_duplicates(subset="patientId").reset_index(drop=True)
print("De-duplicated shape (one row per patient):", class_info.shape)

print("\nClass distribution (full patient population):")
print(class_info["class"].value_counts())
print("\nClass distribution (%):")
print((class_info["class"].value_counts(normalize=True) * 100).round(2))


**Class definitions:**

| Class label in CSV | Final project class name | Clinical meaning |
|---|---|---|
| `Lung Opacity` | **Pneumonia** | Opacity finding consistent with pneumonia |
| `Normal` | **Normal** | No abnormal finding |
| `No Lung Opacity / Not Normal` | **Not Normal / No Lung Opacity** | An abnormality is present, radiologist-adjudicated as *not* pneumonia |


In [ ]:
# Map to the final three-class label used throughout this notebook.
# IMPORTANT: this is a 3-way mapping, not the interim's 2-way collapse.
CLASS_NAMES = ["Normal", "Not Normal / No Lung Opacity", "Pneumonia"]
CLASS_TO_LABEL = {
    "Normal": 0,
    "No Lung Opacity / Not Normal": 1,
    "Lung Opacity": 2,
}

class_info["label"] = class_info["class"].map(CLASS_TO_LABEL)
assert class_info["label"].isna().sum() == 0, "Unmapped class values found — inspect `class` column."

print("Label mapping check:")
print(class_info[["class", "label"]].drop_duplicates().sort_values("label"))


In [ ]:
# Optional stratified subsample (see SAMPLE_SIZE in config above).
if SAMPLE_SIZE is not None:
    working_df, _ = train_test_split(
        class_info,
        train_size=SAMPLE_SIZE,
        stratify=class_info["class"],
        random_state=RANDOM_STATE,
    )
    working_df = working_df.reset_index(drop=True)
else:
    working_df = class_info.copy()

print("Working dataset shape:", working_df.shape)
print("Number of classes:", working_df["class"].nunique())
print("\nClass distribution (working set):")
print(working_df["class"].value_counts())


**Verify image storage and inspect DICOM metadata**

DICOM (`.dcm`) files store pixel data alongside header metadata (modality,
patient orientation, bits allocated, etc.). Before building a loader, we
confirm the file naming convention (`<patientId>.dcm`) and inspect a sample
file's structure — carrying forward the interim notebook's loading
convention.

In [ ]:
# Sanity-check that image files exist for the naming convention we assume,
# and inspect one DICOM header + pixel array shape/dtype.
sample_patient_id = working_df.iloc[0]["patientId"]
sample_path = os.path.join(IMAGE_DIR, f"{sample_patient_id}.dcm")
assert os.path.exists(sample_path), (
    f"Expected DICOM file not found at {sample_path}. "
    "Check IMAGE_DIR and the stage_2_train_images folder structure."
)

sample_dcm = pydicom.dcmread(sample_path)
print("DICOM header (selected fields):")
print("  Modality:           ", sample_dcm.get("Modality"))
print("  PhotometricInterp:  ", sample_dcm.get("PhotometricInterpretation"))
print("  Rows x Columns:     ", sample_dcm.Rows, "x", sample_dcm.Columns)
print("  BitsAllocated:      ", sample_dcm.get("BitsAllocated"))

pix = sample_dcm.pixel_array
print("\nPixel array shape:", pix.shape, "| dtype:", pix.dtype,
      "| min:", pix.min(), "| max:", pix.max())


**Observations (fill in numbers after running against your actual data):**

- The full patient population is far larger and more balanced across the
  three classes than the interim notebook's 500-image subsample — this
  matters, because the interim's binary model's total failure (0.00 recall
  on the minority class) was substantially a product of extreme imbalance
  in that small sample, not just the model architecture.
- Each patient in this dataset has exactly one diagnostic label, so
  patient-level and image-level splitting are equivalent here — no
  patient-level leakage risk from multiple studies per patient (unlike the
  bounding-box table, which has multiple *rows* per Lung Opacity patient;
  the de-duplication step above accounts for that).
- Record the actual DICOM pixel array shape and bit depth from the cell
  above — chest X-rays are frequently stored at higher resolution
  (e.g. 1024×1024) than the target model input size, so resizing is
  necessary regardless of architecture choice (addressed in Data
  Preprocessing).


---
# 2. Exploratory Data Analysis

## 2.1 Random images from each class

**Why this step is needed:** visual inspection lets us sanity-check that
labels are plausible and gives an early sense of how visually distinguishable
the three classes are — particularly whether *Not Normal / No Lung Opacity*
is visually separable from *Pneumonia*, which the interim notebook
identified as the harder distinction.

**What is being done:** five random images are sampled per class and
displayed with their labels.

In [ ]:
def load_dicom_grayscale(patient_id: str, image_dir: str = IMAGE_DIR) -> np.ndarray:
    """Read a DICOM file and return an 8-bit single-channel (grayscale) array,
    min-max normalized per-image to the full 0-255 display range."""
    path = os.path.join(image_dir, f"{patient_id}.dcm")
    dcm = pydicom.dcmread(path)
    pixel_array = dcm.pixel_array.astype(np.float32)
    pixel_array = (
        (pixel_array - pixel_array.min())
        / (pixel_array.max() - pixel_array.min() + 1e-8)
        * 255.0
    ).astype(np.uint8)
    return pixel_array


In [ ]:
N_PER_CLASS = 5
rng = np.random.RandomState(RANDOM_STATE)

fig, axes = plt.subplots(3, N_PER_CLASS, figsize=(3 * N_PER_CLASS, 10))

for row_idx, cls_name in enumerate(CLASS_TO_LABEL.keys()):
    subset = working_df[working_df["class"] == cls_name]
    sampled = subset.sample(n=min(N_PER_CLASS, len(subset)), random_state=rng)
    for col_idx, (_, r) in enumerate(sampled.iterrows()):
        img = load_dicom_grayscale(r["patientId"])
        ax = axes[row_idx, col_idx]
        ax.imshow(img, cmap="gray")
        ax.set_title(cls_name, fontsize=9)
        ax.axis("off")

plt.suptitle("Random Sample Images per Class (Raw DICOM Pixel Data)", fontsize=13)
plt.tight_layout()
plt.savefig("eda_sample_images_per_class.png", dpi=150, bbox_inches="tight")
plt.show()


**Observations (confirm against your actual output):**

- All three classes are grayscale chest radiographs; no color information is
  present at the source (`PhotometricInterpretation` confirmed in Data
  Overview), which supports the rubric's grayscale-conversion step being a
  format standardization rather than an information-losing transformation.
- Visually compare the *Pneumonia* and *Not Normal / No Lung Opacity* rows
  above — per the interim notebook's own analysis, some *Not Normal / No
  Lung Opacity* cases show visible opacities or irregularities that are
  *not* pneumonia-attributable per radiologist adjudication, but can
  resemble the *Pneumonia* row at a glance. This is the expected source of
  class confusion to watch for in the confusion matrices later.
- *Normal* cases should show comparatively uniform, symmetric lung fields
  with no opacity — use this as the visual reference point when reviewing
  misclassifications.


## 2.2 Class Imbalance

**Why this step is needed:** class imbalance materially affects both model
training (a model can achieve high accuracy by favoring the majority class,
exactly what happened in the interim binary model) and evaluation strategy
(accuracy alone becomes misleading).

**What is being done:** class counts and percentages are computed and
visualized. Note this is computed on `working_df`, i.e. reflects whatever
`SAMPLE_SIZE` was configured — reported percentages will match the full
population only if `SAMPLE_SIZE = None`.

In [ ]:
class_counts = working_df["class"].value_counts()
class_pcts = (working_df["class"].value_counts(normalize=True) * 100).round(2)

imbalance_df = pd.DataFrame({
    "Class": class_counts.index,
    "Count": class_counts.values,
    "Percentage": class_pcts[class_counts.index].values,
})
print(imbalance_df.to_string(index=False))


In [ ]:
plt.figure(figsize=(7, 5))
order = working_df["class"].value_counts().index
ax = sns.countplot(data=working_df, x="class", order=order, palette="viridis")
plt.title("Class Distribution — Chest X-Ray Diagnostic Categories")
plt.xlabel("Class")
plt.ylabel("Number of Patients")
plt.xticks(rotation=15, ha="right")

# annotate bars with count and percentage
total = len(working_df)
for p in ax.patches:
    count = int(p.get_height())
    pct = count / total * 100
    ax.annotate(f"{count}\n({pct:.1f}%)",
                (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.savefig("eda_class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()


**Observations (based on the full patient population; re-verify against your
printed output if `SAMPLE_SIZE` is set):**

- The dataset is **imbalanced**: *No Lung Opacity / Not Normal* is the
  largest class (~44%), followed by *Normal* (~33%), with **Pneumonia as
  the minority class (~23%)**.
- This is a materially different — though less extreme — version of the
  imbalance that caused the interim binary CNN to collapse to a
  majority-class predictor (0.00 recall on Pneumonia). The risk applies
  here too: a 3-class model can reach a deceptively high *accuracy* by
  under-predicting the Pneumonia class, while the clinically costliest
  error (missing a true pneumonia case) goes unmeasured by accuracy alone.
- **Implication for later sections:** this motivates (a) reporting
  per-class recall/F1 and the confusion matrix as primary evaluation
  criteria rather than accuracy, alongside macro-F1 to avoid rewarding
  majority-class bias, and (b) evaluating whether class weighting helps
  here — the interim notebook's attempt to fix imbalance via
  `class_weight='balanced'` combined with aggressive threshold tuning
  overcorrected into the opposite failure mode (predicting everything as
  positive), so if class weights are used in Section 4 (CNN from scratch),
  results must be checked for that failure pattern specifically, not just
  for improved recall in isolation.


## 2.3 Additional EDA Observations

**Image dimensions and format**, confirmed from the Data Overview DICOM
header inspection above — record the actual values here once run:

- Source resolution: *(fill in from `sample_dcm.Rows` x `sample_dcm.Columns`
  printed earlier — RSNA chest X-rays are commonly stored at 1024×1024)*
- Bit depth: *(fill in from `BitsAllocated`)*
- All images are single-channel (grayscale) at the source; no RGB images
  exist in this dataset, so the "RGB → grayscale" preprocessing step
  (rubric §7.1) is really a format confirmation + explicit single-channel
  reshape, not a lossy color-to-grayscale conversion.

**Key challenges this EDA surfaces for classification:**

- Class imbalance (above), with the minority class being the clinically
  most important one to detect (Pneumonia) — precision/recall tradeoffs
  need to be evaluated with that asymmetry in mind, not treated as a
  generic multi-class problem.
- Visual similarity between *Pneumonia* and *Not Normal / No Lung Opacity*
  — the three-class problem is harder than either of the binary framings
  the interim notebook explored, by design; that's exactly why the final
  project restores the third class rather than merging it away.
- Source resolution is much higher than typical CNN input sizes, so
  resizing (Section 3) will discard fine detail — worth flagging as a
  limitation in the final report rather than a silent step.


---
# 3. Data Preprocessing

## 3.1 Grayscale Conversion

**Why this step is needed:** the rubric requires an explicit RGB→grayscale
conversion step with before/after visual evidence.

**An honest note on this dataset specifically:** the Data Overview section
already confirmed via `PhotometricInterpretation` that these DICOM images
are natively single-channel grayscale — there is no RGB color data to strip
out (chest X-rays are acquired as grayscale radiographs; there is no
clinical color information to lose). So rather than claim a color-to-
grayscale conversion that isn't actually happening, the transformation shown
below is what our pipeline *actually* does: raw DICOM pixel values (which
can be at arbitrary bit depth and dynamic range) are min-max normalized to
an 8-bit `[0, 255]` range and given an explicit `(H, W, 1)` channel
dimension, matching the shape a Conv2D input layer expects. This is the
real "before → after" transformation for this dataset, and it's shown
below with actual pixel value ranges rather than assumed ones.

In [ ]:
sample_row = working_df.iloc[0]
sample_pid = sample_row["patientId"]
sample_path = os.path.join(IMAGE_DIR, f"{sample_pid}.dcm")

# BEFORE: raw DICOM pixel array, native bit depth / dynamic range
dcm = pydicom.dcmread(sample_path)
raw_pixels = dcm.pixel_array.astype(np.float32)

# AFTER: normalized to 8-bit grayscale with explicit channel dimension
after_pixels = load_dicom_grayscale(sample_pid)          # (H, W), uint8, [0,255]
after_pixels_chan = after_pixels[..., np.newaxis]         # (H, W, 1)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(raw_pixels, cmap="gray")
axes[0].set_title(f"BEFORE — raw DICOM pixels\nrange [{raw_pixels.min():.0f}, {raw_pixels.max():.0f}], dtype {dcm.pixel_array.dtype}")
axes[0].axis("off")

axes[1].imshow(after_pixels_chan.squeeze(), cmap="gray")
axes[1].set_title(f"AFTER — normalized grayscale\nrange [{after_pixels.min()}, {after_pixels.max()}], shape {after_pixels_chan.shape}")
axes[1].axis("off")

plt.suptitle(f"Preprocessing: Before vs. After — Patient {sample_pid}", fontsize=12)
plt.tight_layout()
plt.savefig("preprocessing_before_after.png", dpi=150, bbox_inches="tight")
plt.show()


## 3.2 Image Resizing

**Why this step is needed:** source DICOM resolution (confirmed in Data
Overview) is far larger than any CNN input we'll train — full resolution is
computationally infeasible at the batch sizes/epoch counts required here,
and pretrained backbones expect a specific input size.

**What is being done and why these specific sizes:**
- **128×128** for the from-scratch CNN — matches the interim notebook's
  baseline size, keeping that comparison apples-to-apples and keeping the
  from-scratch model computationally light, which is appropriate since it's
  meant to be a simple baseline, not the final architecture.
- **224×224** for VGG16 and ResNet50 — both networks' ImageNet weights were
  learned at this resolution; transfer-learning quality degrades when input
  size deviates substantially from the pretraining resolution, since early
  conv filters are tuned to that scale of spatial detail.

In [ ]:
source_h, source_w = dcm.Rows, dcm.Columns
print(f"Source DICOM resolution: {source_h} x {source_w}")
print(f"Downsampling ratio for CNN-from-scratch (128x128): "
      f"{source_h/IMG_SIZE_CNN:.1f}x height, {source_w/IMG_SIZE_CNN:.1f}x width")
print(f"Downsampling ratio for transfer models (224x224):  "
      f"{source_h/IMG_SIZE_TRANSFER:.1f}x height, {source_w/IMG_SIZE_TRANSFER:.1f}x width")


In [ ]:
def preprocess_image(patient_id: str, target_size: int, as_rgb: bool,
                      image_dir: str = IMAGE_DIR) -> np.ndarray:
    """Load a DICOM, normalize to 8-bit grayscale, resize, and optionally
    replicate to 3 channels for pretrained-backbone compatibility.

    Returns a uint8 array of shape (target_size, target_size, 1 or 3).
    """
    img = load_dicom_grayscale(patient_id, image_dir)
    img = cv2.resize(img, (target_size, target_size), interpolation=cv2.INTER_AREA)
    if as_rgb:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    else:
        img = img[..., np.newaxis]
    return img


# Sanity check on the sample patient
check_cnn = preprocess_image(sample_pid, IMG_SIZE_CNN, as_rgb=False)
check_transfer = preprocess_image(sample_pid, IMG_SIZE_TRANSFER, as_rgb=True)
print("CNN-from-scratch input shape:", check_cnn.shape)
print("Transfer-model input shape:  ", check_transfer.shape)


## 3.3 Train / Validation / Test Split

**Why this step is needed:** we need isolated splits so validation can guide
training decisions (architecture, hyperparameters, early stopping) while the
test set remains untouched until final evaluation — otherwise reported
performance is optimistic.

**Strategy:** stratified 70/15/15 split on the three-class label, at the
patient level. As confirmed in Data Overview, each patient has exactly one
diagnostic label in this dataset (no repeated studies per patient beyond
the bounding-box duplication already resolved), so patient-level and
image-level splitting are equivalent here and there is no leakage pathway
from a patient appearing in two splits with different images. We still add
an explicit assertion, per good practice, rather than assuming this.

In [ ]:
# First split off the test set, then split remaining into train/val
train_val_df, test_df = train_test_split(
    working_df,
    test_size=TEST_RATIO,
    stratify=working_df["label"],
    random_state=RANDOM_STATE,
)

val_relative_size = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
train_df, val_df = train_test_split(
    train_val_df,
    test_size=val_relative_size,
    stratify=train_val_df["label"],
    random_state=RANDOM_STATE,
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train: {len(train_df)} ({len(train_df)/len(working_df)*100:.1f}%)")
print(f"Val:   {len(val_df)} ({len(val_df)/len(working_df)*100:.1f}%)")
print(f"Test:  {len(test_df)} ({len(test_df)/len(working_df)*100:.1f}%)")

print("\nClass distribution per split (%):")
split_dist = pd.DataFrame({
    "Train": train_df["class"].value_counts(normalize=True) * 100,
    "Val":   val_df["class"].value_counts(normalize=True) * 100,
    "Test":  test_df["class"].value_counts(normalize=True) * 100,
}).round(2)
print(split_dist)


In [ ]:
# Explicit leakage check: no patientId should appear in more than one split
train_ids = set(train_df["patientId"])
val_ids = set(val_df["patientId"])
test_ids = set(test_df["patientId"])

assert train_ids.isdisjoint(val_ids), "Leakage: patient(s) in both train and val!"
assert train_ids.isdisjoint(test_ids), "Leakage: patient(s) in both train and test!"
assert val_ids.isdisjoint(test_ids), "Leakage: patient(s) in both val and test!"
print("No patient-level leakage across train/val/test splits.")


## 3.4 Normalization

**Why this step is needed:** raw pixel values (already rescaled to `[0,255]`
uint8 in the grayscale step) still need to be brought into a numeric range
suitable for gradient-based training.

**What is used and why — two different schemes for two different model
families:**
- **From-scratch CNN:** simple `[0, 1]` rescaling (divide by 255). This
  model has no pretrained prior, so any consistent, gradient-friendly range
  works; `[0,1]` is standard and matches the interim notebook's approach.
- **VGG16 / ResNet50 (transfer learning):** each backbone's official
  `preprocess_input` function (imported above), **not** simple `[0,1]`
  rescaling. These networks were trained with specific per-channel
  normalization statistics from ImageNet (mean subtraction, and for
  ResNet50 additionally BGR channel ordering); using a different
  normalization than what the backbone was pretrained with actively hurts
  transfer performance, since the pretrained filters expect their original
  input distribution.

In [ ]:
def normalize_for_scratch_cnn(img_uint8: np.ndarray) -> np.ndarray:
    """[0,255] uint8 -> [0,1] float32."""
    return img_uint8.astype(np.float32) / 255.0


def normalize_for_vgg16(img_uint8: np.ndarray) -> np.ndarray:
    """[0,255] uint8 RGB -> VGG16's expected input distribution."""
    return vgg16_preprocess(img_uint8.astype(np.float32))


def normalize_for_resnet50(img_uint8: np.ndarray) -> np.ndarray:
    """[0,255] uint8 RGB -> ResNet50's expected input distribution."""
    return resnet50_preprocess(img_uint8.astype(np.float32))


# Demonstrate expected output ranges on the sample image
demo_scratch = normalize_for_scratch_cnn(check_cnn)
demo_vgg = normalize_for_vgg16(check_transfer)
print(f"Scratch-CNN normalization range: [{demo_scratch.min():.3f}, {demo_scratch.max():.3f}]")
print(f"VGG16 normalization range:       [{demo_vgg.min():.2f}, {demo_vgg.max():.2f}] (mean-centered, not [0,1] — expected)")


## 3.5 Data Augmentation

**Why this step is needed:** with a moderate-sized medical imaging dataset
and deep pretrained backbones, overfitting risk is real — augmentation
increases effective training diversity without collecting more data.

**What is used and why — clinically-reasonable transformations only:**
small rotations, shifts, zoom, and brightness jitter simulate realistic
acquisition variation (patient positioning, exposure differences) without
altering diagnostically meaningful structure. **Horizontal flip is
deliberately excluded** — chest anatomy is not left-right symmetric in a way
that should be treated as invariant (heart position, aortic arch, etc.), so
flipping could teach the model incorrect anatomical priors. This carries
forward the interim notebook's reasoning.

**Critically: augmentation is applied to the training set only.** Validation
and test data must reflect the true, un-augmented data distribution the
model will see in deployment — augmenting them would make evaluation
metrics unrepresentative of real-world performance.

In [ ]:
# Implemented as Keras preprocessing layers so they compose cleanly into a
# tf.data pipeline and are only active during training (they no-op at
# inference/eval time when called with training=False, which model.fit/
# .evaluate/.predict handle automatically for layers built this way).
train_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(factor=10/360, fill_mode="constant", fill_value=0),
    tf.keras.layers.RandomTranslation(height_factor=0.05, width_factor=0.05,
                                       fill_mode="constant", fill_value=0),
    tf.keras.layers.RandomZoom(height_factor=0.10, fill_mode="constant", fill_value=0),
    tf.keras.layers.RandomBrightness(factor=0.10, value_range=(0, 255)),
], name="train_augmentation")

# Quick visual check of augmentation on the sample image
demo_img = tf.expand_dims(check_cnn.astype(np.float32), axis=0)  # add batch dim
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
axes[0].imshow(check_cnn.squeeze(), cmap="gray")
axes[0].set_title("Original")
axes[0].axis("off")
for i in range(1, 5):
    aug_img = train_augmentation(demo_img, training=True)[0].numpy().squeeze()
    axes[i].imshow(aug_img, cmap="gray")
    axes[i].set_title(f"Augmented {i}")
    axes[i].axis("off")
plt.suptitle("Data Augmentation — Training Set Only")
plt.tight_layout()
plt.savefig("augmentation_examples.png", dpi=150, bbox_inches="tight")
plt.show()


**Observations:**

- Confirm visually above that augmented samples remain clinically
  plausible (no extreme distortion, no flipped anatomy, no unrealistic
  color/contrast shifts).
- Augmentation is applied inside the training pipeline as Keras layers
  (built in Section 4 when constructing the tf.data pipelines for each
  model), so validation/test pipelines simply omit this step rather than
  needing a separate "no-augmentation" flag threaded through — reducing the
  chance of accidentally leaking augmentation into evaluation data.


---
# 4. Model Building — CNN from Scratch

## 4.1 Building the Input Pipeline

**Why this step is needed:** with ~26.7k images loaded from individual
DICOM files (not a single array that fits comfortably in memory at scale,
especially once we get to 224×224×3 for the transfer-learning sections), a
`tf.data` pipeline that loads and preprocesses on the fly — rather than
pre-loading everything into a NumPy array — keeps memory usage bounded and
lets us reuse the same pipeline builder across all model sections by just
changing `target_size`, `as_rgb`, and `normalize_fn`.

**What is being done:** `build_tf_dataset` wraps `preprocess_image` (Section
3.2) and the appropriate normalization function (Section 3.4) in a
`tf.py_function` (needed because `pydicom` isn't a native TensorFlow op),
batches, optionally shuffles and augments, and prefetches. Augmentation
(Section 3.5) is applied **after batching and only when `augment=True`** —
we will only ever pass `augment=True` for training pipelines.

In [ ]:
def build_tf_dataset(df, target_size, as_rgb, normalize_fn,
                      batch_size=32, augment=False, shuffle=False):
    """Build a tf.data pipeline that loads DICOMs by patientId, preprocesses,
    and batches. Reused for every model in this notebook by varying
    target_size / as_rgb / normalize_fn."""
    patient_ids = df["patientId"].values
    labels = df["label"].values.astype(np.int32)
    channels = 3 if as_rgb else 1

    def _load(pid, label):
        pid_str = pid.numpy().decode("utf-8")
        img = preprocess_image(pid_str, target_size, as_rgb)
        img = normalize_fn(img)
        return img.astype(np.float32), label

    def _tf_load(pid, label):
        img, label = tf.py_function(_load, [pid, label], [tf.float32, tf.int32])
        img.set_shape([target_size, target_size, channels])
        label.set_shape([])
        return img, label

    ds = tf.data.Dataset.from_tensor_slices((patient_ids, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(df), seed=RANDOM_STATE)
    ds = ds.map(_tf_load, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size)
    if augment:
        ds = ds.map(lambda x, y: (train_augmentation(x, training=True), y),
                     num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


In [ ]:
# Note: DICOM decoding (pydicom) is CPU-bound Python code running inside
# tf.py_function, so it won't be as fast as a pure-TF image pipeline. If
# training is slow on your runtime, consider pre-decoding once to .npy or
# .png on disk and building a simpler pipeline from those — not done here
# to keep this notebook self-contained and runnable directly against the
# raw DICOM files as provided.

BATCH_SIZE = 32

train_ds_cnn = build_tf_dataset(
    train_df, target_size=IMG_SIZE_CNN, as_rgb=False,
    normalize_fn=normalize_for_scratch_cnn,
    batch_size=BATCH_SIZE, augment=True, shuffle=True,
)
val_ds_cnn = build_tf_dataset(
    val_df, target_size=IMG_SIZE_CNN, as_rgb=False,
    normalize_fn=normalize_for_scratch_cnn,
    batch_size=BATCH_SIZE, augment=False, shuffle=False,
)
test_ds_cnn = build_tf_dataset(
    test_df, target_size=IMG_SIZE_CNN, as_rgb=False,
    normalize_fn=normalize_for_scratch_cnn,
    batch_size=BATCH_SIZE, augment=False, shuffle=False,
)

# Confirm shapes on one batch
for xb, yb in train_ds_cnn.take(1):
    print("Batch image shape:", xb.shape, "| Batch label shape:", yb.shape)
    print("Label dtype:", yb.dtype, "| Sample labels:", yb.numpy()[:8])


## 4.2 Class Weights

**Why this step is needed:** Section 2.2 confirmed Pneumonia is the
minority class (~22.5%). A from-scratch CNN has no pretrained prior to fall
back on, so it's the model most exposed to majority-class bias — the same
failure mode that produced 0.00 recall on the interim's binary model.

**What is different this time:** the interim notebook's fix attempt failed
because it combined class weights *with* aggressive decision-threshold
tuning on a binary sigmoid output, which is a narrow, easy-to-overcorrect
lever. Here, with a 3-class softmax + `argmax` decision rule, there is no
equivalent threshold to over-tune — class weights are the only lever being
pulled, which makes their effect easier to isolate and evaluate on their
own merits via the per-class recall/F1 reported after training, rather than
compounding two interacting fixes at once.

In [ ]:
class_weight_values = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"],
)
class_weight_dict = dict(enumerate(class_weight_values))
print("Class weights (0=Normal, 1=Not Normal/No Opacity, 2=Pneumonia):")
print(class_weight_dict)


## 4.3 CNN Architecture

**Why this step is needed:** a from-scratch CNN establishes a baseline
performance floor before introducing pretrained backbones — it tells us how
much transfer learning actually buys us over a simple model trained only on
this dataset.

**Architecture, block by block:**
- **3 convolutional blocks** (32 → 64 → 128 filters), each: `Conv2D` (3×3,
  ReLU) to learn local spatial features → `BatchNormalization` to stabilize
  training → `MaxPooling2D` (2×2) to downsample and build translation
  invariance. Filter count increases with depth, following the standard
  pattern of learning more abstract, higher-level features as spatial
  resolution shrinks.
- **Flatten → Dense(256, ReLU)** to combine spatial features into a single
  representation.
- **Dropout(0.5)** for regularization — needed given a moderate-sized
  medical dataset and a reasonably large dense layer, both of which invite
  overfitting.
- **Output: `Dense(3, activation="softmax")`** — three-class probability
  distribution, one unit per class, replacing the interim's single sigmoid
  unit.

In [ ]:
def build_cnn_from_scratch(input_shape=(IMG_SIZE_CNN, IMG_SIZE_CNN, 1), num_classes=3):
    model = Sequential([
        Conv2D(32, (3, 3), activation="relu", padding="same", input_shape=input_shape),
        BatchNormalization(),
        MaxPooling2D((2, 2)),

        Conv2D(64, (3, 3), activation="relu", padding="same"),
        BatchNormalization(),
        MaxPooling2D((2, 2)),

        Conv2D(128, (3, 3), activation="relu", padding="same"),
        BatchNormalization(),
        MaxPooling2D((2, 2)),

        Flatten(),
        Dense(256, activation="relu"),
        Dropout(0.5),
        Dense(num_classes, activation="softmax"),
    ], name="cnn_from_scratch")
    return model


cnn_scratch = build_cnn_from_scratch()
cnn_scratch.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",   # integer labels (0,1,2), not one-hot
    metrics=["accuracy"],
)
cnn_scratch.summary()


## 4.4 Training

**Callbacks used and why:**
- `EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)`
  — stops training once validation loss stops improving, and reverts to the
  best-performing weights rather than the final epoch's, guarding against
  overfitting on a dataset this size.
- `ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3)` — halves
  the learning rate when validation loss plateaus, allowing finer
  convergence without manually re-tuning the learning rate.
- `ModelCheckpoint` — persists the best model to disk during training, as a
  safeguard independent of `restore_best_weights`.

In [ ]:
EPOCHS = 30

callbacks_scratch = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
    ModelCheckpoint("cnn_scratch_best.keras", monitor="val_loss", save_best_only=True),
]

history_scratch = cnn_scratch.fit(
    train_ds_cnn,
    validation_data=val_ds_cnn,
    epochs=EPOCHS,
    class_weight=class_weight_dict,
    callbacks=callbacks_scratch,
    verbose=2,
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(history_scratch.history["accuracy"], label="Train")
axes[0].plot(history_scratch.history["val_accuracy"], label="Validation")
axes[0].set_title("CNN from Scratch — Accuracy over Epochs")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

axes[1].plot(history_scratch.history["loss"], label="Train")
axes[1].plot(history_scratch.history["val_loss"], label="Validation")
axes[1].set_title("CNN from Scratch — Loss over Epochs")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()

plt.tight_layout()
plt.savefig("cnn_scratch_learning_curves.png", dpi=150, bbox_inches="tight")
plt.show()


**Observations (fill in after running):**

- Compare train vs. validation curves: a widening gap (train accuracy
  climbing well above validation) indicates overfitting — expected to some
  degree from a from-scratch model on this data volume, watch whether
  `EarlyStopping` triggered before it became severe.
- Note the epoch at which early stopping triggered (if it did) and whether
  `ReduceLROnPlateau` fired — both are visible in the training log output
  above.


## 4.5 Test Set Evaluation

**Why this step is needed:** validation performance guided training
decisions (early stopping, LR schedule), so it can't be treated as an
unbiased performance estimate — the test set, untouched until now, gives
that. Per the rubric's medical-ML guidance, accuracy alone is reported
alongside macro-averaged and per-class precision/recall/F1, since
class-wise performance (especially Pneumonia recall) matters more than
aggregate accuracy for this use case.

In [ ]:
y_test_true = np.concatenate([y.numpy() for _, y in test_ds_cnn], axis=0)
y_test_pred_probs = cnn_scratch.predict(test_ds_cnn)
y_test_pred = np.argmax(y_test_pred_probs, axis=1)

test_accuracy = accuracy_score(y_test_true, y_test_pred)
test_precision_macro = precision_score(y_test_true, y_test_pred, average="macro", zero_division=0)
test_recall_macro = recall_score(y_test_true, y_test_pred, average="macro", zero_division=0)
test_f1_macro = f1_score(y_test_true, y_test_pred, average="macro", zero_division=0)

print(f"Test Accuracy:        {test_accuracy:.2f}")
print(f"Test Precision (macro): {test_precision_macro:.2f}")
print(f"Test Recall (macro):    {test_recall_macro:.2f}")
print(f"Test F1 (macro):        {test_f1_macro:.2f}")

print("\nFull classification report:\n")
print(classification_report(y_test_true, y_test_pred, target_names=CLASS_NAMES, zero_division=0))


In [ ]:
cm_scratch = confusion_matrix(y_test_true, y_test_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm_scratch, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title("Confusion Matrix — CNN from Scratch (Test Set)")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.savefig("cnn_scratch_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()


**Class-wise confusion discussion (fill in against the actual matrix above):**

Read the confusion matrix as three specific pairwise questions, since each
has a different clinical cost:

- **Pneumonia vs. Normal:** the most clinically serious confusion in either
  direction — a Pneumonia case predicted as Normal is a missed diagnosis
  (false negative); check this cell of the matrix specifically, not just
  overall Pneumonia recall.
- **Pneumonia vs. Not Normal/No Lung Opacity:** the confusion the EDA
  flagged as visually likely, since both can present with lung
  abnormalities. A high count here would validate that concern and is the
  main justification for why this project restores the third class instead
  of using the interim's binary framing — a binary model would have
  silently absorbed this exact confusion into "correct" negative
  predictions.
- **Normal vs. Not Normal/No Lung Opacity:** likely the least clinically
  costly confusion of the three (neither indicates pneumonia), but still
  worth checking that the model isn't just using this pair as a dumping
  ground for uncertain predictions.

This baseline's numbers become the reference point for Section 5 onward —
transfer learning is only worth its added complexity if it measurably
improves on Pneumonia recall and macro-F1 here, not just aggregate
accuracy.


---
# 5. Transfer Learning — VGG16 (Base Model)

## 5.1 Rationale

**Why transfer learning:** the from-scratch CNN (Section 4) learns
low-level visual features (edges, textures, simple shapes) entirely from
this dataset alone. VGG16, pretrained on ImageNet, already has those
low-level filters well-developed from millions of natural images — reusing
them means the model only needs to learn how to *recombine* existing
features into pneumonia-relevant patterns, rather than learning edge/shape
detection from scratch on a comparatively small medical dataset.

**Why VGG16 specifically:** carried forward from the interim notebook's own
proposed next step (Section on file, not overridden here) — its simple,
uniform 3×3-conv-block structure also makes it a natural first pretrained
backbone to establish a transfer-learning baseline before adding
architectural complexity (the "enhanced" version) or fine-tuning.

**Experiment A — base transfer learning:** VGG16's convolutional base is
used purely as a **frozen feature extractor** — none of its pretrained
weights are updated. Only a new classification head, sized for our 3
classes, is trained. This isolates "how good are ImageNet features for this
task, with zero adaptation" as a baseline before fine-tuning (Section 7)
asks the deeper question of whether *adapting* some of those features
helps further.

In [ ]:
train_ds_vgg16 = build_tf_dataset(
    train_df, target_size=IMG_SIZE_TRANSFER, as_rgb=True,
    normalize_fn=normalize_for_vgg16,
    batch_size=BATCH_SIZE, augment=True, shuffle=True,
)
val_ds_vgg16 = build_tf_dataset(
    val_df, target_size=IMG_SIZE_TRANSFER, as_rgb=True,
    normalize_fn=normalize_for_vgg16,
    batch_size=BATCH_SIZE, augment=False, shuffle=False,
)
test_ds_vgg16 = build_tf_dataset(
    test_df, target_size=IMG_SIZE_TRANSFER, as_rgb=True,
    normalize_fn=normalize_for_vgg16,
    batch_size=BATCH_SIZE, augment=False, shuffle=False,
)

for xb, yb in train_ds_vgg16.take(1):
    print("Batch image shape:", xb.shape, "| dtype:", xb.dtype)
    print("Pixel range (VGG16-preprocessed, mean-centered):",
          f"[{xb.numpy().min():.1f}, {xb.numpy().max():.1f}]")


## 5.2 Architecture — Frozen VGG16 + New Classification Head

- **Base:** `VGG16(weights="imagenet", include_top=False, input_shape=(224,224,3))`
  — `include_top=False` drops VGG16's original 1000-class ImageNet
  classification head, keeping only the convolutional feature extractor.
- **`base.trainable = False`** — freezes every pretrained weight; gradients
  will not flow into the VGG16 layers at all during this experiment.
- **`GlobalAveragePooling2D`** — condenses each feature map to a single
  value per channel. Chosen over `Flatten` because it produces far fewer
  parameters into the dense head (reducing overfitting risk on a
  comparatively small dataset) and is the standard choice when adapting a
  pretrained conv backbone to a new head.
- **`Dense(128, relu) → Dropout(0.3) → Dense(3, softmax)`** — a compact new
  classification head; kept intentionally simple here since Section 6
  ("Enhanced" architecture) is where a deliberately larger/deeper head is
  introduced and justified as the point of comparison.

In [ ]:
def build_vgg16_base_model(input_shape=(IMG_SIZE_TRANSFER, IMG_SIZE_TRANSFER, 3), num_classes=3):
    base = VGG16(weights="imagenet", include_top=False, input_shape=input_shape)
    base.trainable = False  # freeze — Experiment A uses VGG16 purely as a feature extractor

    inputs = Input(shape=input_shape)
    x = base(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation="relu")(x)
    x = Dropout(0.3)(x)
    outputs = Dense(num_classes, activation="softmax")(x)

    model = Model(inputs, outputs, name="vgg16_base_transfer")
    return model, base


vgg16_base_model, vgg16_backbone = build_vgg16_base_model()
vgg16_base_model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
vgg16_base_model.summary()

trainable_params = sum(np.prod(v.shape) for v in vgg16_base_model.trainable_weights)
frozen_params = sum(np.prod(v.shape) for v in vgg16_base_model.non_trainable_weights)
print(f"\nTrainable params: {trainable_params:,} | Frozen (VGG16) params: {frozen_params:,}")


In [ ]:
callbacks_vgg16_base = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
    ModelCheckpoint("vgg16_base_best.keras", monitor="val_loss", save_best_only=True),
]

history_vgg16_base = vgg16_base_model.fit(
    train_ds_vgg16,
    validation_data=val_ds_vgg16,
    epochs=EPOCHS,
    class_weight=class_weight_dict,
    callbacks=callbacks_vgg16_base,
    verbose=2,
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(history_vgg16_base.history["accuracy"], label="Train")
axes[0].plot(history_vgg16_base.history["val_accuracy"], label="Validation")
axes[0].set_title("VGG16 Base — Accuracy over Epochs")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Accuracy"); axes[0].legend()

axes[1].plot(history_vgg16_base.history["loss"], label="Train")
axes[1].plot(history_vgg16_base.history["val_loss"], label="Validation")
axes[1].set_title("VGG16 Base — Loss over Epochs")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss"); axes[1].legend()

plt.tight_layout()
plt.savefig("vgg16_base_learning_curves.png", dpi=150, bbox_inches="tight")
plt.show()


## 5.3 Test Set Evaluation

In [ ]:
y_test_true_vgg = np.concatenate([y.numpy() for _, y in test_ds_vgg16], axis=0)
y_test_pred_probs_vgg = vgg16_base_model.predict(test_ds_vgg16)
y_test_pred_vgg = np.argmax(y_test_pred_probs_vgg, axis=1)

print(f"Test Accuracy:          {accuracy_score(y_test_true_vgg, y_test_pred_vgg):.2f}")
print(f"Test Precision (macro): {precision_score(y_test_true_vgg, y_test_pred_vgg, average='macro', zero_division=0):.2f}")
print(f"Test Recall (macro):    {recall_score(y_test_true_vgg, y_test_pred_vgg, average='macro', zero_division=0):.2f}")
print(f"Test F1 (macro):        {f1_score(y_test_true_vgg, y_test_pred_vgg, average='macro', zero_division=0):.2f}")

print("\nFull classification report:\n")
print(classification_report(y_test_true_vgg, y_test_pred_vgg, target_names=CLASS_NAMES, zero_division=0))


In [ ]:
cm_vgg16_base = confusion_matrix(y_test_true_vgg, y_test_pred_vgg)
plt.figure(figsize=(6, 5))
sns.heatmap(cm_vgg16_base, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title("Confusion Matrix — VGG16 Base Transfer Learning (Test Set)")
plt.xlabel("Predicted Label"); plt.ylabel("True Label")
plt.tight_layout()
plt.savefig("vgg16_base_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()


**Observations (compare directly against the Section 4 CNN-from-scratch
numbers once both have run on the same `SAMPLE_SIZE`):**

- Did macro-F1 and Pneumonia-class recall improve over the from-scratch
  CNN? That's the specific question this experiment answers — if not,
  frozen ImageNet features may not be transferring well to X-ray imagery
  (a known risk: ImageNet is natural photos, not radiographs, so the
  domain gap can limit how useful frozen low-level filters are).
- Check whether the train/validation gap is narrower or wider than the
  scratch CNN's — frozen backbones typically overfit less (far fewer
  trainable parameters), which should show up as a smaller gap here.


---
# 6. Enhanced Architecture — VGG16 + Additional Layers

**Why this step is needed:** the rubric requires going beyond a plain
frozen-backbone-plus-simple-head setup — adding deliberate architectural
capacity on top of the pretrained features, to test whether a richer head
extracts more value from VGG16's representations than Section 5's compact
one did.

**What's added and why:**
- An extra `Conv2D(256, 3x3) + BatchNormalization` layer **on the backbone's
  output feature maps**, before pooling — lets the model learn a
  task-specific refinement of VGG16's generic ImageNet features rather than
  pooling them immediately, at the cost of extra parameters.
- A deeper dense head (`256 → 64` with `BatchNormalization` and two dropout
  layers) instead of Section 5's single `Dense(128)` — more representational
  capacity, offset by more aggressive dropout (0.4, 0.3) since deeper heads
  overfit more readily on a moderate-sized dataset.
- Backbone remains **frozen**, same as Section 5 — this experiment isolates
  the effect of head capacity alone; fine-tuning (unfreezing backbone
  layers) is introduced separately in Section 7 so the two effects aren't
  conflated.

In [ ]:
def build_vgg16_enhanced_model(input_shape=(IMG_SIZE_TRANSFER, IMG_SIZE_TRANSFER, 3), num_classes=3):
    base = VGG16(weights="imagenet", include_top=False, input_shape=input_shape)
    base.trainable = False

    inputs = Input(shape=input_shape)
    x = base(inputs, training=False)
    x = Conv2D(256, (3, 3), activation="relu", padding="same")(x)
    x = BatchNormalization()(x)
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation="relu")(x)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)
    x = Dense(64, activation="relu")(x)
    x = Dropout(0.3)(x)
    outputs = Dense(num_classes, activation="softmax")(x)

    model = Model(inputs, outputs, name="vgg16_enhanced")
    return model, base


vgg16_enhanced_model, _ = build_vgg16_enhanced_model()
vgg16_enhanced_model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
vgg16_enhanced_model.summary()


In [ ]:
callbacks_vgg16_enh = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
    ModelCheckpoint("vgg16_enhanced_best.keras", monitor="val_loss", save_best_only=True),
]

history_vgg16_enh = vgg16_enhanced_model.fit(
    train_ds_vgg16, validation_data=val_ds_vgg16, epochs=EPOCHS,
    class_weight=class_weight_dict, callbacks=callbacks_vgg16_enh, verbose=2,
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(history_vgg16_enh.history["accuracy"], label="Train")
axes[0].plot(history_vgg16_enh.history["val_accuracy"], label="Validation")
axes[0].set_title("VGG16 Enhanced — Accuracy"); axes[0].legend()
axes[1].plot(history_vgg16_enh.history["loss"], label="Train")
axes[1].plot(history_vgg16_enh.history["val_loss"], label="Validation")
axes[1].set_title("VGG16 Enhanced — Loss"); axes[1].legend()
plt.tight_layout()
plt.savefig("vgg16_enhanced_learning_curves.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
y_test_true_vgg_enh = np.concatenate([y.numpy() for _, y in test_ds_vgg16], axis=0)
y_test_pred_probs_vgg_enh = vgg16_enhanced_model.predict(test_ds_vgg16)
y_test_pred_vgg_enh = np.argmax(y_test_pred_probs_vgg_enh, axis=1)

print(classification_report(y_test_true_vgg_enh, y_test_pred_vgg_enh, target_names=CLASS_NAMES, zero_division=0))

cm_vgg16_enh = confusion_matrix(y_test_true_vgg_enh, y_test_pred_vgg_enh)
plt.figure(figsize=(6, 5))
sns.heatmap(cm_vgg16_enh, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title("Confusion Matrix — VGG16 Enhanced (Test Set)")
plt.xlabel("Predicted Label"); plt.ylabel("True Label")
plt.tight_layout()
plt.savefig("vgg16_enhanced_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()


**Observations:** compare macro-F1 and Pneumonia recall against Section 5's base VGG16 — did the added capacity help, or mainly increase the train/validation gap (overfitting) without a corresponding test-set gain?

---
# 7. Fine-Tuning — VGG16

**Why this step is needed:** Sections 5–6 never updated VGG16's pretrained
weights at all. Fine-tuning tests whether letting the *later* (more
task-specific) convolutional layers adapt to chest X-ray features —  while
keeping *early* layers (generic edges/textures, broadly useful across
domains) frozen — improves performance further.

**What is frozen vs. unfrozen:** VGG16's last convolutional block
(`block5_conv1` through `block5_conv3`, plus `block5_pool`) is unfrozen;
blocks 1–4 remain frozen. Rationale: early conv blocks learn generic
low-level features that transfer well regardless of domain, so unfreezing
them risks destroying useful pretrained knowledge with too few
domain-specific examples; block 5's filters are the most
task/dataset-specific in the network, making them the right layers to
adapt.

**Learning rate:** dropped to **1e-5** (100x lower than the 1e-3 used for
the frozen-backbone experiments) — fine-tuning pretrained weights requires
small updates to avoid catastrophically overwriting useful pretrained
representations, a well-established practice for this technique.

**Starting point:** fine-tuning continues from the **enhanced** architecture
(Section 6), since it's the stronger head design of the two frozen
experiments — assuming Section 6 outperformed Section 5 on validation; if
your results show otherwise, swap in the Section 5 architecture here
instead and note why in the report.

In [ ]:
# Unfreeze block5 only; keep blocks 1-4 frozen
vgg16_backbone_ft = vgg16_enhanced_model.layers[1]  # the VGG16 base layer within the functional model
vgg16_backbone_ft.trainable = True

for layer in vgg16_backbone_ft.layers:
    if layer.name.startswith("block5"):
        layer.trainable = True
    else:
        layer.trainable = False

print("Unfrozen VGG16 layers:")
for layer in vgg16_backbone_ft.layers:
    if layer.trainable:
        print(" -", layer.name)


In [ ]:
vgg16_finetuned_model = vgg16_enhanced_model  # continuing training on the same model object
vgg16_finetuned_model.compile(
    optimizer=Adam(learning_rate=1e-5),   # much lower LR for fine-tuning
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

trainable_params_ft = sum(np.prod(v.shape) for v in vgg16_finetuned_model.trainable_weights)
print(f"Trainable params after unfreezing block5: {trainable_params_ft:,}")

callbacks_vgg16_ft = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7),
    ModelCheckpoint("vgg16_finetuned_best.keras", monitor="val_loss", save_best_only=True),
]

FT_EPOCHS = 15  # fewer epochs than the initial training phase - fine-tuning converges faster
                # from an already-good starting point, and is more overfitting-prone

history_vgg16_ft = vgg16_finetuned_model.fit(
    train_ds_vgg16, validation_data=val_ds_vgg16, epochs=FT_EPOCHS,
    class_weight=class_weight_dict, callbacks=callbacks_vgg16_ft, verbose=2,
)


In [ ]:
y_test_true_vgg_ft = np.concatenate([y.numpy() for _, y in test_ds_vgg16], axis=0)
y_test_pred_probs_vgg_ft = vgg16_finetuned_model.predict(test_ds_vgg16)
y_test_pred_vgg_ft = np.argmax(y_test_pred_probs_vgg_ft, axis=1)

print(classification_report(y_test_true_vgg_ft, y_test_pred_vgg_ft, target_names=CLASS_NAMES, zero_division=0))

cm_vgg16_ft = confusion_matrix(y_test_true_vgg_ft, y_test_pred_vgg_ft)
plt.figure(figsize=(6, 5))
sns.heatmap(cm_vgg16_ft, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title("Confusion Matrix — VGG16 Fine-Tuned (Test Set)")
plt.xlabel("Predicted Label"); plt.ylabel("True Label")
plt.tight_layout()
plt.savefig("vgg16_finetuned_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()


**Observations:** did unfreezing block5 improve Pneumonia recall/macro-F1 over the frozen Enhanced version? Watch for overfitting signs (val loss rising while train loss falls) — fine-tuning is the experiment most prone to it given the added trainable capacity.

---
# 8. Transfer Learning — ResNet50 (Base Model)

**Why ResNet50 as the second architecture:** carried forward from the
interim notebook's proposed improvements, alongside VGG16. Architecturally
distinct from VGG16 in a way that makes the comparison meaningful rather
than redundant: ResNet50 uses residual/skip connections, allowing it to be
much deeper (50 layers vs. VGG16's 16) without the vanishing-gradient
problems that plague plain deep stacks — worth testing whether that
additional depth and different inductive bias helps or hurts on X-ray
imagery specifically.

Same Experiment-A structure as Section 5: frozen backbone, compact new
head, `ResNet50`'s own `preprocess_input` (not VGG16's — ResNet50 was
trained with different normalization statistics and BGR ordering).

In [ ]:
train_ds_resnet = build_tf_dataset(
    train_df, target_size=IMG_SIZE_TRANSFER, as_rgb=True,
    normalize_fn=normalize_for_resnet50,
    batch_size=BATCH_SIZE, augment=True, shuffle=True,
)
val_ds_resnet = build_tf_dataset(
    val_df, target_size=IMG_SIZE_TRANSFER, as_rgb=True,
    normalize_fn=normalize_for_resnet50,
    batch_size=BATCH_SIZE, augment=False, shuffle=False,
)
test_ds_resnet = build_tf_dataset(
    test_df, target_size=IMG_SIZE_TRANSFER, as_rgb=True,
    normalize_fn=normalize_for_resnet50,
    batch_size=BATCH_SIZE, augment=False, shuffle=False,
)

def build_resnet50_base_model(input_shape=(IMG_SIZE_TRANSFER, IMG_SIZE_TRANSFER, 3), num_classes=3):
    base = ResNet50(weights="imagenet", include_top=False, input_shape=input_shape)
    base.trainable = False

    inputs = Input(shape=input_shape)
    x = base(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation="relu")(x)
    x = Dropout(0.3)(x)
    outputs = Dense(num_classes, activation="softmax")(x)

    model = Model(inputs, outputs, name="resnet50_base_transfer")
    return model, base


resnet50_base_model, resnet50_backbone = build_resnet50_base_model()
resnet50_base_model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
resnet50_base_model.summary()


In [ ]:
callbacks_resnet_base = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
    ModelCheckpoint("resnet50_base_best.keras", monitor="val_loss", save_best_only=True),
]

history_resnet_base = resnet50_base_model.fit(
    train_ds_resnet, validation_data=val_ds_resnet, epochs=EPOCHS,
    class_weight=class_weight_dict, callbacks=callbacks_resnet_base, verbose=2,
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(history_resnet_base.history["accuracy"], label="Train")
axes[0].plot(history_resnet_base.history["val_accuracy"], label="Validation")
axes[0].set_title("ResNet50 Base — Accuracy"); axes[0].legend()
axes[1].plot(history_resnet_base.history["loss"], label="Train")
axes[1].plot(history_resnet_base.history["val_loss"], label="Validation")
axes[1].set_title("ResNet50 Base — Loss"); axes[1].legend()
plt.tight_layout()
plt.savefig("resnet50_base_learning_curves.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
y_test_true_resnet = np.concatenate([y.numpy() for _, y in test_ds_resnet], axis=0)
y_test_pred_probs_resnet = resnet50_base_model.predict(test_ds_resnet)
y_test_pred_resnet = np.argmax(y_test_pred_probs_resnet, axis=1)

print(classification_report(y_test_true_resnet, y_test_pred_resnet, target_names=CLASS_NAMES, zero_division=0))

cm_resnet_base = confusion_matrix(y_test_true_resnet, y_test_pred_resnet)
plt.figure(figsize=(6, 5))
sns.heatmap(cm_resnet_base, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title("Confusion Matrix — ResNet50 Base (Test Set)")
plt.xlabel("Predicted Label"); plt.ylabel("True Label")
plt.tight_layout()
plt.savefig("resnet50_base_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()


**Observations:** compare against both the scratch CNN and VGG16 results so far — does ResNet50's deeper, residual architecture transfer better or worse than VGG16's simpler structure for this domain?

---
# 9. Enhanced Architecture — ResNet50 + Additional Layers

Same rationale as Section 6 (VGG16 Enhanced): additional Conv2D+BatchNorm "
                 "on the backbone output, plus a deeper dense head, backbone still frozen — "
                 "isolating the effect of head capacity before fine-tuning is introduced.

In [ ]:
def build_resnet50_enhanced_model(input_shape=(IMG_SIZE_TRANSFER, IMG_SIZE_TRANSFER, 3), num_classes=3):
    base = ResNet50(weights="imagenet", include_top=False, input_shape=input_shape)
    base.trainable = False

    inputs = Input(shape=input_shape)
    x = base(inputs, training=False)
    x = Conv2D(256, (3, 3), activation="relu", padding="same")(x)
    x = BatchNormalization()(x)
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation="relu")(x)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)
    x = Dense(64, activation="relu")(x)
    x = Dropout(0.3)(x)
    outputs = Dense(num_classes, activation="softmax")(x)

    model = Model(inputs, outputs, name="resnet50_enhanced")
    return model, base


resnet50_enhanced_model, _ = build_resnet50_enhanced_model()
resnet50_enhanced_model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
resnet50_enhanced_model.summary()


In [ ]:
callbacks_resnet_enh = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
    ModelCheckpoint("resnet50_enhanced_best.keras", monitor="val_loss", save_best_only=True),
]

history_resnet_enh = resnet50_enhanced_model.fit(
    train_ds_resnet, validation_data=val_ds_resnet, epochs=EPOCHS,
    class_weight=class_weight_dict, callbacks=callbacks_resnet_enh, verbose=2,
)


In [ ]:
y_test_true_resnet_enh = np.concatenate([y.numpy() for _, y in test_ds_resnet], axis=0)
y_test_pred_probs_resnet_enh = resnet50_enhanced_model.predict(test_ds_resnet)
y_test_pred_resnet_enh = np.argmax(y_test_pred_probs_resnet_enh, axis=1)

print(classification_report(y_test_true_resnet_enh, y_test_pred_resnet_enh, target_names=CLASS_NAMES, zero_division=0))

cm_resnet_enh = confusion_matrix(y_test_true_resnet_enh, y_test_pred_resnet_enh)
plt.figure(figsize=(6, 5))
sns.heatmap(cm_resnet_enh, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title("Confusion Matrix — ResNet50 Enhanced (Test Set)")
plt.xlabel("Predicted Label"); plt.ylabel("True Label")
plt.tight_layout()
plt.savefig("resnet50_enhanced_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()


---
# 10. Fine-Tuning — ResNet50

**What is unfrozen:** ResNet50's final residual block group (`conv5_block1`
through `conv5_block3`) — analogous to unfreezing VGG16's block5 in Section
7: the last, most task-specific block, while earlier residual blocks
(generic features) stay frozen. Same fine-tuning learning rate (1e-5) and
same rationale — small updates to avoid destroying pretrained
representations.

In [ ]:
resnet50_backbone_ft = resnet50_enhanced_model.layers[1]
resnet50_backbone_ft.trainable = True

for layer in resnet50_backbone_ft.layers:
    if layer.name.startswith("conv5_block"):
        layer.trainable = True
    else:
        layer.trainable = False

print("Unfrozen ResNet50 layers (sample):")
unfrozen = [l.name for l in resnet50_backbone_ft.layers if l.trainable]
print(f"{len(unfrozen)} layers unfrozen, e.g.: {unfrozen[:5]}")


In [ ]:
resnet50_finetuned_model = resnet50_enhanced_model
resnet50_finetuned_model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks_resnet_ft = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7),
    ModelCheckpoint("resnet50_finetuned_best.keras", monitor="val_loss", save_best_only=True),
]

history_resnet_ft = resnet50_finetuned_model.fit(
    train_ds_resnet, validation_data=val_ds_resnet, epochs=FT_EPOCHS,
    class_weight=class_weight_dict, callbacks=callbacks_resnet_ft, verbose=2,
)


In [ ]:
y_test_true_resnet_ft = np.concatenate([y.numpy() for _, y in test_ds_resnet], axis=0)
y_test_pred_probs_resnet_ft = resnet50_finetuned_model.predict(test_ds_resnet)
y_test_pred_resnet_ft = np.argmax(y_test_pred_probs_resnet_ft, axis=1)

print(classification_report(y_test_true_resnet_ft, y_test_pred_resnet_ft, target_names=CLASS_NAMES, zero_division=0))

cm_resnet_ft = confusion_matrix(y_test_true_resnet_ft, y_test_pred_resnet_ft)
plt.figure(figsize=(6, 5))
sns.heatmap(cm_resnet_ft, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title("Confusion Matrix — ResNet50 Fine-Tuned (Test Set)")
plt.xlabel("Predicted Label"); plt.ylabel("True Label")
plt.tight_layout()
plt.savefig("resnet50_finetuned_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()


---
# 11. Model Comparison

**Why this step is needed:** individual sections evaluated each model in
isolation; this section consolidates all seven experiments into one table
so trade-offs (accuracy vs. complexity vs. Pneumonia-recall) are visible
side by side, per rubric §13/§34.

**What is being done:** metrics already computed in each section above are
gathered into a single comparison table — no re-computation, no new
numbers, purely consolidation. If you re-run sections out of order or clear
outputs, re-run the cells above first so these variables are populated.

In [ ]:
def summarize_experiment(name, y_true, y_pred, architecture, pretrained,
                          input_size, trainable_params, notes=""):
    return {
        "Model": name,
        "Architecture": architecture,
        "Pretrained Weights": pretrained,
        "Input Size": input_size,
        "Trainable Params": trainable_params,
        "Test Accuracy": round(accuracy_score(y_true, y_pred), 2),
        "Precision (macro)": round(precision_score(y_true, y_pred, average="macro", zero_division=0), 2),
        "Recall (macro)": round(recall_score(y_true, y_pred, average="macro", zero_division=0), 2),
        "F1 (macro)": round(f1_score(y_true, y_pred, average="macro", zero_division=0), 2),
        "Pneumonia Recall": round(
            recall_score(y_true, y_pred, labels=[CLASS_TO_LABEL["Lung Opacity"]],
                         average="macro", zero_division=0), 2),
        "Notes": notes,
    }


experiment_records = [
    summarize_experiment(
        "CNN from Scratch", y_test_true, y_test_pred,
        "3-block CNN", "None", f"{IMG_SIZE_CNN}x{IMG_SIZE_CNN}",
        cnn_scratch.count_params(), "Baseline"),
    summarize_experiment(
        "VGG16 Base", y_test_true_vgg, y_test_pred_vgg,
        "VGG16 (frozen) + simple head", "ImageNet", f"{IMG_SIZE_TRANSFER}x{IMG_SIZE_TRANSFER}",
        sum(np.prod(v.shape) for v in vgg16_base_model.trainable_weights), "Experiment A"),
    summarize_experiment(
        "VGG16 Enhanced", y_test_true_vgg_enh, y_test_pred_vgg_enh,
        "VGG16 (frozen) + deep head", "ImageNet", f"{IMG_SIZE_TRANSFER}x{IMG_SIZE_TRANSFER}",
        sum(np.prod(v.shape) for v in vgg16_enhanced_model.trainable_weights), "Added Conv+deeper head"),
    summarize_experiment(
        "VGG16 Fine-Tuned", y_test_true_vgg_ft, y_test_pred_vgg_ft,
        "VGG16 (block5 unfrozen) + deep head", "ImageNet", f"{IMG_SIZE_TRANSFER}x{IMG_SIZE_TRANSFER}",
        sum(np.prod(v.shape) for v in vgg16_finetuned_model.trainable_weights), "block5 unfrozen, lr=1e-5"),
    summarize_experiment(
        "ResNet50 Base", y_test_true_resnet, y_test_pred_resnet,
        "ResNet50 (frozen) + simple head", "ImageNet", f"{IMG_SIZE_TRANSFER}x{IMG_SIZE_TRANSFER}",
        sum(np.prod(v.shape) for v in resnet50_base_model.trainable_weights), "Experiment A"),
    summarize_experiment(
        "ResNet50 Enhanced", y_test_true_resnet_enh, y_test_pred_resnet_enh,
        "ResNet50 (frozen) + deep head", "ImageNet", f"{IMG_SIZE_TRANSFER}x{IMG_SIZE_TRANSFER}",
        sum(np.prod(v.shape) for v in resnet50_enhanced_model.trainable_weights), "Added Conv+deeper head"),
    summarize_experiment(
        "ResNet50 Fine-Tuned", y_test_true_resnet_ft, y_test_pred_resnet_ft,
        "ResNet50 (conv5 unfrozen) + deep head", "ImageNet", f"{IMG_SIZE_TRANSFER}x{IMG_SIZE_TRANSFER}",
        sum(np.prod(v.shape) for v in resnet50_finetuned_model.trainable_weights), "conv5 unfrozen, lr=1e-5"),
]

comparison_df = pd.DataFrame(experiment_records).sort_values("F1 (macro)", ascending=False).reset_index(drop=True)
comparison_df


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
metrics_to_plot = ["Test Accuracy", "Precision (macro)", "Recall (macro)", "F1 (macro)"]
comparison_df.set_index("Model")[metrics_to_plot].plot(kind="bar", ax=ax)
plt.title("Model Comparison — Test Set Metrics")
plt.ylabel("Score")
plt.xticks(rotation=30, ha="right")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("model_comparison_chart.png", dpi=150, bbox_inches="tight")
plt.show()


**Discussion points (fill in against actual results):**

- Which model has the **highest Pneumonia recall** — not just highest
  accuracy? In a screening context, this is arguably the single most
  important column in this table, since a missed Pneumonia case (false
  negative) is more clinically costly than a false positive that a
  radiologist reviews and dismisses.
- Compare train/validation gaps (from each section's learning curves) to
  flag overfitting — a model with strong test metrics but a large train/val
  gap is a less trustworthy generalizer than one with closer-tracking
  curves, even at similar test scores.
- Compare **trainable parameter counts** — did fine-tuning's added capacity
  translate into proportionally better results, or mostly added overfitting
  risk and training time for a marginal gain?


---
# 12. Best Model Selection

**Selection criteria (in priority order for this medical screening use
case):**

1. **Pneumonia recall** — minimizing missed pneumonia cases is the primary
   clinical objective of a screening-support tool.
2. **Macro-F1** — balances performance across all three classes rather than
   over-indexing on any one, including the majority classes.
3. **Generalization gap** (train vs. validation/test) — a model that
   generalizes reliably is more trustworthy in deployment than one that
   memorized training data, even at a similar test score.
4. **Model complexity / inference cost** — relevant for deployment
   feasibility (Section 16), particularly if this tool needs to run in
   resource-constrained healthcare settings, per the business context.

**This is not automated** — the cell below suggests a candidate by ranking
on Pneumonia recall then macro-F1, but the actual selection in your report
should be a written judgment call referencing the specific trade-offs in
the comparison table above, not just the top row of a sort.

In [ ]:
candidate_ranking = comparison_df.sort_values(
    ["Pneumonia Recall", "F1 (macro)"], ascending=[False, False]
).reset_index(drop=True)

print("Candidates ranked by Pneumonia Recall, then macro-F1:")
print(candidate_ranking[["Model", "Pneumonia Recall", "F1 (macro)", "Test Accuracy", "Trainable Params"]])

# Set this manually after reviewing the full comparison table and your
# report's written rationale - do not treat the top row above as an
# automatic final answer.
BEST_MODEL_NAME = None  # e.g. "VGG16 Fine-Tuned"
BEST_MODEL_OBJECT = None  # e.g. vgg16_finetuned_model
BEST_MODEL_TARGET_SIZE = None  # e.g. IMG_SIZE_TRANSFER
BEST_MODEL_AS_RGB = None  # True for VGG16/ResNet50 models, False for scratch CNN
BEST_MODEL_NORMALIZE_FN = None  # e.g. normalize_for_vgg16

assert BEST_MODEL_NAME is not None, (
    "Set BEST_MODEL_NAME/OBJECT/TARGET_SIZE/AS_RGB/NORMALIZE_FN above after "
    "reviewing the comparison table and writing your rationale."
)


---
# 13. Model Serialization, Reload, and Sample Inference

In [ ]:
BEST_MODEL_PATH = "best_model.keras"
BEST_MODEL_OBJECT.save(BEST_MODEL_PATH)
print(f"Saved {BEST_MODEL_NAME} to {BEST_MODEL_PATH}")

# Reload and verify
reloaded_model = tf.keras.models.load_model(BEST_MODEL_PATH)
print("Reload successful. Model summary:")
reloaded_model.summary()


In [ ]:
# Sample inference on a handful of test-set images, using the SAME
# preprocessing pipeline as training (critical - mismatched preprocessing
# between training and inference is a common, silent source of bad
# real-world performance).
N_SAMPLES = 8
sample_test_rows = test_df.sample(n=N_SAMPLES, random_state=RANDOM_STATE).reset_index(drop=True)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, row in sample_test_rows.iterrows():
    img = preprocess_image(row["patientId"], BEST_MODEL_TARGET_SIZE, BEST_MODEL_AS_RGB)
    img_norm = BEST_MODEL_NORMALIZE_FN(img)
    img_batch = np.expand_dims(img_norm, axis=0)

    probs = reloaded_model.predict(img_batch, verbose=0)[0]
    pred_label = np.argmax(probs)

    display_img = load_dicom_grayscale(row["patientId"])
    axes[i].imshow(display_img, cmap="gray")
    axes[i].axis("off")
    title = (f"Actual: {CLASS_NAMES[row['label']]}\n"
             f"Predicted: {CLASS_NAMES[pred_label]} ({probs[pred_label]*100:.1f}%)")
    color = "green" if pred_label == row["label"] else "red"
    axes[i].set_title(title, fontsize=9, color=color)

    print(f"Patient {row['patientId']}: actual={CLASS_NAMES[row['label']]}, "
          f"predicted={CLASS_NAMES[pred_label]}")
    for cname, p in zip(CLASS_NAMES, probs):
        print(f"    {cname}: {p*100:.2f}%")
    print()

plt.suptitle(f"Sample Inference — {BEST_MODEL_NAME}", fontsize=13)
plt.tight_layout()
plt.savefig("sample_inference.png", dpi=150, bbox_inches="tight")
plt.show()


---
# 14. Deployment Preparation

The reloaded model above (`best_model.keras`) plus the preprocessing
function it depends on (`preprocess_image` + the specific normalization
function used, recorded in `BEST_MODEL_NORMALIZE_FN`/`BEST_MODEL_AS_RGB`/
`BEST_MODEL_TARGET_SIZE`) are exactly what the Streamlit app needs. Export
a small config file so the app doesn't have to guess these values.

In [ ]:
import json as _json

deployment_config = {
    "model_name": BEST_MODEL_NAME,
    "target_size": BEST_MODEL_TARGET_SIZE,
    "as_rgb": bool(BEST_MODEL_AS_RGB),
    "normalize": "vgg16" if BEST_MODEL_NORMALIZE_FN is normalize_for_vgg16
                 else "resnet50" if BEST_MODEL_NORMALIZE_FN is normalize_for_resnet50
                 else "rescale",
    "class_names": CLASS_NAMES,
}

with open("deployment_config.json", "w") as f:
    _json.dump(deployment_config, f, indent=2)

print("Wrote deployment_config.json:")
print(_json.dumps(deployment_config, indent=2))
print("\nCopy best_model.keras and deployment_config.json into the model/ "
      "folder of the deployment repo (see app.py / README provided separately).")
